In [ ]:
import numpy as np

from scipy.stats import spearmanr

import pandas as pd

import itertools

import random

import tool

from dataSet import W2V_weighted_DataSet_v2
from data.pipData import pipe_data, prepare_data, prepare_data_with_intonation, separate_text_intonation
from modelSGNS import SGNS_OneEmbWeighted

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from collections import Counter

from copy import deepcopy

from tqdm import tqdm

import traceback

import os
import hashlib

In [ ]:
def see_corelation_between_df(df1:pd.DataFrame, df2:pd.DataFrame)-> None:
    common_words = df1.index.intersection(df2.index)
    assert len(common_words) > 3, "no suffisent word in common"
    
    common_words = sorted(common_words)
    
    m1_sub = df1.loc[common_words, common_words].values
    m2_sub = df2.loc[common_words, common_words].values
    
    # Get only triangle of 
    upper_indices = np.triu_indices(len(common_words), k=1)
    vec1 = m1_sub[upper_indices]
    vec2 = m2_sub[upper_indices]
    
    corr, p_value = spearmanr(vec1, vec2)
    return corr, p_value

def remake_pair(dataset:W2V_weighted_DataSet_v2, new_windows:int, nb_neg:int) -> W2V_weighted_DataSet_v2:
    dataset.K = nb_neg
    dataset.context_size = new_windows
    dataset._make_pairs_positif()
    return dataset

def train(model:nn.Module, optimizer:torch.optim.Optimizer, loader:DataLoader, nb_epoch:int, writer=None):
    model.train()
    for epoch in range(nb_epoch):
        total_loss = 0
        for sentence_nb, (centers, pos, negs, intonation_t, intonation_c) in enumerate(loader):
            optimizer.zero_grad()
            centers = centers.to("cuda")
            pos = pos.to("cuda")
            negs = negs.to("cuda")
            
            weight_loss = torch.minimum(intonation_t, intonation_c).to("cuda")
            
            loss:torch.Tensor = model((centers, pos, negs, weight_loss))
            loss.backward()
            total_loss += loss.item()
            optimizer.step()

        avg_loss = total_loss / len(loader)
        
        if writer is not None:
            writer.add_scalar("Loss/train", avg_loss, epoch)
        
    return avg_loss

def create_all_data(dataset:W2V_weighted_DataSet_v2, all_param:list[dict]) -> dict:
    all_dataset = {}
    intonations = dataset.intonations
    for param in all_param:
        into = tool.normalize_range_center(intonations, range_normalize=param['range_norm'], 
                                           center=param['center_norm'])
        new_dataset:W2V_weighted_DataSet_v2 = deepcopy(dataset)
        new_dataset.K = param['nb_neg']
        new_dataset.context_size = param['windows_size']
        new_dataset.intonations = into
        new_dataset.pairs = new_dataset._make_pairs_positif()
        key_tuple = tuple(param.values())
        all_dataset[key_tuple] = new_dataset
    return all_dataset

def compute_all_cosine(vectors:torch.tensor):
    vectors_norm = F.normalize(vectors, p=2, dim=1)
    sim_matrix = torch.mm(vectors_norm, vectors_norm.t())
    return sim_matrix

def next_save_df(df:pd.DataFrame, folder:str="."):
    os.makedirs(folder, exist_ok=True)
    nb_file = len(os.listdir(folder))
    df.to_csv(path_or_buf=f"{folder}/MS_{nb_file}.csv", float_format="%.8f", index=True)
    return f"{folder}/MS_{nb_file}.csv"

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # if you use multi-GPU
    # For absolute reproducibility (may slow down training slightly):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
{
    "param" : {
        "dim" : 3,
        "lr" : 0.03,
        'nb_epoch': 25,
        'optim': "adam",
        'windows_size': 5,
        'nb_neg' : 10,
        'range_norm' : 1.9,
        'center_norm' : 1
    },
    "results": {
            'loss' : [0.0001, 0.0002],
            'corelationGoogle' : [(0.34, 0.0001), (0.43 , 0.0001)],
            'anisotropie' : [{}]
        }
}

In [ ]:
data = prepare_data_with_intonation(
    file_path="./data/GoodNightGorilla_Intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={# specific to corpus 
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",},
    stop_words=["s", "n't"],
    break_line=False
)
texts, intonations = separate_text_intonation(data)

dataset = W2V_weighted_DataSet_v2(sentences=texts, intonations=intonations,
                        nb_neg=0, window_size=0)
all_token = []
for sentence in dataset.tokens:
    all_token.extend(sentence)
    
freq = Counter(all_token)
freq_list = [freq.get(i, 0) for i in range(len(dataset.decoder.keys()))]

unigram = torch.tensor([f**0.75 for f in freq_list], dtype=torch.float) # Formule utilisé par le Word 2 vec original
unigram = unigram / unigram.sum()

dataset.unigram_dist = unigram

In [ ]:
all_data_param = {
    'windows_size': [2, 4, 6],
    'nb_neg' : [5, 10, 20],
    'range_norm' : [1.9, 1.75,  1.5, 1.25, 0.5],
    'center_norm' : [1],
}

all_model_param = {
    "dim" : range(3, 20, 2),
    "lr" : [0.0003, 0.003, 0.03],
    "batch_size":[8, 16, 32, 64],
    "shuffle":[False],
    'nb_epoch': [25],
    "sparse": [False],
    'optim': [torch.optim.Adam]
}

keys_model = all_model_param.keys()
values_model = all_model_param.values()
combinations_param_model = list(itertools.product(*values_model))
list_of_model_param = [dict(zip(keys_model, combo)) for combo in  combinations_param_model]

keys_data = all_data_param.keys()
values_data = all_data_param.values()
combinations_param_data = list(itertools.product(*values_data))
list_of_dataset_param = [dict(zip(keys_data, combo)) for combo in  combinations_param_data]

In [ ]:
all_data = create_all_data(dataset=dataset, all_param=list_of_dataset_param)

In [ ]:
# for combo in combinations_param_model:
#     params = 
#     print(f"Testing model with: {params}")
#     remake_pair(dataset, new_windows=params["windows_size"], nb_neg=params["nb_neg"])
    
    
#     tool.analyser_anisotropie_advanced()

In [ ]:
model_to_test:type[SGNS_OneEmbWeighted] = SGNS_OneEmbWeighted
nb_instance = 5
log_dir_base = "runs/grid_search_experiment"
list_of_seeds = [1, 2, 3, 4, 5]

total_iterations = len(list_of_dataset_param) * len(list_of_model_param)
results = []
failed_runs = []

with tqdm(total=total_iterations, desc="Grid Search Progress") as pbar:
    for data_param in list_of_dataset_param:
        dataset_key = tuple(data_param.values())
        try:
            current_dataset:W2V_weighted_DataSet_v2 = all_data[dataset_key]
        except KeyError:
            print(f"CRITICAL ERROR: Could not find dataset for {data_param}")
            pbar.update(len(list_of_model_param)) 
            continue
        
        for model_param in list_of_model_param:
            full_params = {**data_param, **model_param}
            tb_params = full_params.copy()
            if "optim" in tb_params:
                tb_params["optim"] = tb_params["optim"].__name__
                
            matrix_similarities = {}
                        
            param_str = str(sorted(tb_params.items()))
            param_hash = hashlib.md5(param_str.encode()).hexdigest()[:8]
                
            try:
                list_of_emb = []
                for instance in range(nb_instance):
                    current_seed = list_of_seeds[instance]
                    set_all_seeds(current_seed)
                    run_name = f"{log_dir_base}/{param_hash}/run_{instance}"
                    writer = SummaryWriter(log_dir=run_name)             
                    loader = DataLoader(dataset=current_dataset, batch_size=model_param["batch_size"], 
                                        shuffle=model_param["shuffle"])
                    
                    model = model_to_test(emb_size=len(current_dataset.vocab),
                                        embedding_dimension=model_param["dim"], 
                                        sparse=model_param["sparse"], device="cuda")
                    
                    class_opti:type[torch.optim.Optimizer] = model_param["optim"]
                    opti = class_opti(model.parameters(), lr=model_param["lr"])
                    
                    score = train(loader=loader, model=model, nb_epoch=model_param['nb_epoch'], optimizer=opti, writer=writer)
                    
                    print(tb_params)
                    writer.add_hparams(
                        hparam_dict=tb_params,
                        metric_dict={'hparam/score': score}
                    )
                    
                    writer.close()
                    
                    vectors = model.word_emb.weight.detach()
                    res = compute_all_cosine(vectors)
                    res = res.detach().cpu().numpy()
                    df = pd.DataFrame(res, columns=dataset.encoder.keys(), index=dataset.encoder.keys())
                    list_of_emb.append(df)
                    matrix_similarities[instance] = next_save_df(df, 
    f"grid_search_results/matrix_similarity/{tb_params["shuffle"]}/{tb_params["sparse"]}/{tb_params["nb_epoch"]}/{tb_params["optim"]}/{tb_params["batch_size"]}/{tb_params["dim"]}/{tb_params["lr"]}/{tb_params["center_norm"]}/{tb_params["range_norm"]}/{tb_params["nb_neg"]}/{tb_params["windows_size"]}"
    )
                correlation_scores = []
                p_values = []
                for i in range(nb_instance):
                    for j in range(i + 1, nb_instance):
                        df1 = list_of_emb[i]
                        df2 = list_of_emb[j]
                        
                        # Calculate correlation between two different random seeds
                        c, p = see_corelation_between_df(df1, df2)
                        
                        correlation_scores.append(c)
                        p_values.append(p)
                mean_correlation = np.mean(c)
                mean_p_value = np.mean(p)
                std_correlation = np.std(correlation_scores)
                
                agg_run_name = f"{log_dir_base}/{param_hash}/run_aggregate"
                agg_writer = SummaryWriter(log_dir=agg_run_name)
                
                agg_writer.add_hparams(
                    hparam_dict=tb_params,
                    metric_dict={
                        'hparam/stability_correlation': mean_correlation,
                        'hparam/stability_p_value': mean_p_value,
                        'hparam/std_dev': std_correlation,
                    })
                agg_writer.add_scalar("Stability/Mean_Correlation", mean_correlation, 0)
                agg_writer.close()
        
                    
            except Exception as e:
                # If an error occurs, we catch it here
                error_message = str(e)
                print(f"\n[!] Error with params: {full_params}")
                print(f"    Reason: {error_message}")
                
                failed_runs.append({
                    'params': full_params,
                    'error': error_message
                })
                
            finally:
                # This block runs whether the code succeeded or failed
                pbar.update(1) # Advance the progress bar by 1

In [ ]:
for param_number in range(1): 
    print(f"for param : {results[param_number]["params"]}") 
    for i in range(nb_instance):
        print(results[param_number]["scores"][i])
        

for i in range(nb_instance):
    df1 = results[param_number]["ms"][i]
    for j in range(nb_instance):
        
        df2 = results[param_number]["ms"][j]
        print(see_corelation_between_df(df1, df2))